# Pipeline de avaliação — multi-modelo × multi-edital

Roda o agente em todas as combinações `(modelo, edital)` definidas, salva um xlsx
por combinação em `resultados/` e exibe sumário consolidado.

**Proteções contra desastre:**
1. Validação de preços antes de rodar (pega `PRECOS` faltando).
2. Smoke test: 1 pergunta por combinação antes de soltar as 50.
3. Limite de gasto por combinação — corta no meio se estourar.
4. Salvamento garantido em qualquer erro: o que foi medido até a falha vai pro xlsx.
5. Sumário final lista sucessos, parciais e falhas separadamente.

In [1]:
import os, sys, time, glob
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.agent import build_agent, ask
from langchain_core.callbacks import BaseCallbackHandler

print(f"cwd:  {Path.cwd()}")
print(f"root: {ROOT}")

cwd:  /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/evals
root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant


## TokenTracker — captura tokens + cache

In [2]:
class TokenTracker(BaseCallbackHandler):
    def __init__(self):
        self.reset()

    def reset(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.cache_read_tokens = 0
        self.cache_write_tokens = 0
        self.n_calls = 0

    def on_llm_end(self, response, **kwargs):
        self.n_calls += 1

        usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
        if usage:
            self.input_tokens  += usage.get("prompt_tokens", 0) or 0
            self.output_tokens += usage.get("completion_tokens", 0) or 0
            ds_hit = usage.get("prompt_cache_hit_tokens") or 0
            details = usage.get("prompt_tokens_details") or {}
            oai_hit = details.get("cached_tokens") or 0
            self.cache_read_tokens += max(ds_hit, oai_hit)
            return

        for gen_list in response.generations:
            for gen in gen_list:
                msg = getattr(gen, "message", None)
                um = getattr(msg, "usage_metadata", None) if msg else None
                if um:
                    self.input_tokens  += um.get("input_tokens", 0) or 0
                    self.output_tokens += um.get("output_tokens", 0) or 0
                    details = um.get("input_token_details") or {}
                    self.cache_read_tokens  += details.get("cache_read", 0) or 0
                    self.cache_write_tokens += details.get("cache_creation", 0) or 0
                    break

## Configuração

Comente os índices que não quer rodar. `LIMITE_USD_POR_COMBO` é teto sobre estimativa
pessimista (sem cache) — real será sempre menor.

In [3]:
MODELOS = [
    ## (provider, modelo)
    # ("openai",    "gpt-4o-mini"),
    # ("deepseek",  "deepseek-v4-flash"),
    # ("openai",    "gpt-5.4-mini"),
    # ("deepseek",  "deepseek-v4-pro"),
    # ("anthropic", "claude-haiku-4-5"),
    # --- modelos caros — descomente quando for rodar ---
    # ("openai",    "gpt-5.4"),
    # ("anthropic", "claude-sonnet-4-6"),
    ("anthropic", "claude-opus-4-7"),
    # ("openai",    "gpt-5.5"),
]

EDITAIS = [
    # ("bndes",     1),
    ("cvm",       2),
    ("petrobras", 3),
]

# USD por 1M tokens. Ordem: (input_miss, input_cached, output).
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.075,  0.60),
    "openai/gpt-5.4-mini":            (0.25,  0.125,  2.00),
    "openai/gpt-5.4":                 (1.25,  0.625, 10.00),
    "openai/gpt-5.5":                 (5.00,  2.500, 30.00),
    "deepseek/deepseek-v4-flash":     (0.14,  0.028,  0.28),
    "deepseek/deepseek-v4-pro":       (1.74,  0.145,  3.48),
    "anthropic/claude-haiku-4-5":     (1.00,  1.000,  5.00),
    "anthropic/claude-sonnet-4-6":    (3.00,  3.000, 15.00),
    "anthropic/claude-opus-4-7":      (5.00,  5.000, 25.00),
}

LIMITE_USD_POR_COMBO = 20.00

PERGUNTAS_DIR  = Path("perguntas")
RESULTADOS_DIR = Path("resultados")
RESULTADOS_DIR.mkdir(exist_ok=True)

print(f"Modelos ativos:        {len(MODELOS)}")
print(f"Editais:               {len(EDITAIS)}")
print(f"Total de combinações:  {len(MODELOS) * len(EDITAIS)}")
print(f"Teto por combinação:   ${LIMITE_USD_POR_COMBO}")

Modelos ativos:        1
Editais:               2
Total de combinações:  2
Teto por combinação:   $20.0


## Validação 1 — preços (gratuita, não chama API)

In [4]:
faltando = [f"{p}/{m}" for p, m in MODELOS if f"{p}/{m}" not in PRECOS]
if faltando:
    raise RuntimeError(f"Modelos sem preço em PRECOS: {faltando}")
print(f"OK — todos os {len(MODELOS)} modelos têm preço configurado.")

OK — todos os 1 modelos têm preço configurado.


## Helpers

In [5]:
def slug(modelo: str) -> str:
    return modelo.replace("/", "_").replace(":", "_")


def custo_estimado(in_tok: int, out_tok: int, p_in: float, p_out: float) -> float:
    """Teto: trata todo input como cache miss."""
    return in_tok / 1_000_000 * p_in + out_tok / 1_000_000 * p_out


def custo_real(in_tok: int, out_tok: int, cache_read: int,
               p_in: float, p_in_cached: float, p_out: float) -> float:
    miss = max(in_tok - cache_read, 0)
    return (
        miss / 1_000_000 * p_in
        + cache_read / 1_000_000 * p_in_cached
        + out_tok / 1_000_000 * p_out
    )


def salvar_xlsx(provider, modelo, edital_nome, resultados, sufixo=""):
    df_r = pd.DataFrame(resultados)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    nome = f"{edital_nome}__{provider}__{slug(modelo)}__{ts}{sufixo}.xlsx"
    arq = RESULTADOS_DIR / nome
    df_r.to_excel(arq, index=False)
    return arq


def apagar_antigos(provider, modelo, edital_nome):
    padrao = str(RESULTADOS_DIR / f"{edital_nome}__{provider}__{slug(modelo)}__*.xlsx")
    n = 0
    for antigo in glob.glob(padrao):
        Path(antigo).unlink()
        n += 1
    return n

In [6]:
def rodar_combo(provider, modelo, edital_nome, edital_id, limite_usd=LIMITE_USD_POR_COMBO):
    """Roda N perguntas. Retorna (xlsx_path, erro_fatal_ou_None).
    
    Erros em perguntas individuais vão pra coluna `erro` e a combinação continua.
    Estouro de orçamento ou erro fora do loop interrompem e salvam parcial."""
    chave = f"{provider}/{modelo}"
    p_in, p_in_cached, p_out = PRECOS[chave]

    arq_perguntas = PERGUNTAS_DIR / f"{edital_nome}.xlsx"
    df_q = pd.read_excel(arq_perguntas)
    df_q = df_q.dropna(subset=["pergunta"])
    df_q = df_q[df_q["pergunta"].str.strip() != ""].reset_index(drop=True)

    agente = build_agent(provider=provider, model=modelo)
    tracker = TokenTracker()
    agente.llm       = agente.llm.with_config(callbacks=[tracker])
    agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

    resultados = []
    custo_estim_acum = 0.0
    erro_fatal = None

    for i, row in df_q.iterrows():
        pergunta = row["pergunta"]
        t0 = time.time()
        erro, resposta = None, None

        tracker.reset()
        try:
            resposta = ask(agent=agente, question=pergunta, chat_history=[], edital_id_ativo=edital_id)
        except Exception as e:
            erro = str(e)

        in_tok     = tracker.input_tokens
        out_tok    = tracker.output_tokens
        cache_read = tracker.cache_read_tokens
        n_calls    = tracker.n_calls
        latencia   = round(time.time() - t0, 2)

        c_estim = custo_estimado(in_tok, out_tok, p_in, p_out)
        c_real  = custo_real(in_tok, out_tok, cache_read, p_in, p_in_cached, p_out)
        custo_estim_acum += c_estim

        resultados.append({
            "id":                row["id"],
            "categoria":         row.get("categoria", ""),
            "pergunta":          pergunta,
            "resposta":          resposta,
            "input_tokens":      in_tok,
            "output_tokens":     out_tok,
            "total_tokens":      in_tok + out_tok,
            "cache_read_tokens": cache_read,
            "cache_miss_tokens": max(in_tok - cache_read, 0),
            "n_invocacoes":      n_calls,
            "custo_usd":         round(c_estim, 6),
            "custo_real_usd":    round(c_real, 6),
            "latencia_s":        latencia,
            "erro":              erro,
        })

        flag = "ERRO" if erro else "ok"
        cache_pct = (cache_read / in_tok * 100) if in_tok else 0
        print(f"      [{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
              f"calls={n_calls}  in={in_tok:>5} (cache {cache_pct:>4.0f}%)  "
              f"out={out_tok:>4}  acum=${custo_estim_acum:.4f}")

        if custo_estim_acum > limite_usd:
            erro_fatal = f"Estouro na pergunta {i+1}: ${custo_estim_acum:.4f} > ${limite_usd}"
            break

    sufixo = "__PARCIAL" if erro_fatal else ""
    arq = salvar_xlsx(provider, modelo, edital_nome, resultados, sufixo) if resultados else None
    return arq, erro_fatal

## Validação 2 — smoke test

Uma pergunta por combinação. Se algum modelo tiver armadilha (thinking mode default,
credencial faltando, formato de tool diferente), explode aqui em vez de em 50
perguntas. Se qualquer combo falhar, **a célula levanta erro** e o loop principal
não roda.

In [7]:
print("=== SMOKE TEST ===\n")

smoke_falhas = []

for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        print(f"  [test] {provider}/{modelo} × {edital_nome} ... ", end="", flush=True)
        try:
            agente = build_agent(provider=provider, model=modelo)
            tracker = TokenTracker()
            agente.llm       = agente.llm.with_config(callbacks=[tracker])
            agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

            resp = ask(agent=agente, question="Qual o salário inicial?",
                       chat_history=[], edital_id_ativo=edital_id)

            if not resp or len(resp.strip()) < 20:
                raise ValueError(f"resposta vazia/curta: {resp!r}")
            print(f"ok ({tracker.input_tokens} in / {tracker.output_tokens} out / {tracker.n_calls} calls)")
        except Exception as e:
            smoke_falhas.append(f"{provider}/{modelo} × {edital_nome}: {e}")
            print(f"FALHOU\n           {e}")

if smoke_falhas:
    raise RuntimeError(
        f"\n{len(smoke_falhas)} combinação(ões) falhou(ram). Corrija ou comente:\n  - " +
        "\n  - ".join(smoke_falhas)
    )

print(f"\nTodas as {len(MODELOS) * len(EDITAIS)} combinações passaram.")

=== SMOKE TEST ===

  [test] anthropic/claude-opus-4-7 × cvm ... Carregando BGE-M3 (primeira vez pode demorar)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26165.47it/s]


ok (13721 in / 215 out / 2 calls)
  [test] anthropic/claude-opus-4-7 × petrobras ... ok (15234 in / 278 out / 2 calls)

Todas as 2 combinações passaram.


## Pipeline principal

Sobrescreve xlsx antigos. Erro numa combinação não para as outras. Sumário
no fim separa sucessos, parciais (estouro) e falhas.

In [8]:
sucessos, parciais, falhas = [], [], []

for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        n = apagar_antigos(provider, modelo, edital_nome)
        if n:
            print(f"\n[del]  {n} xlsx antigo(s) de {provider}/{modelo} × {edital_nome}")
        print(f"\n[run]  {provider}/{modelo} × {edital_nome} (id={edital_id})")

        try:
            arq, erro_fatal = rodar_combo(provider, modelo, edital_nome, edital_id)
        except Exception as e:
            falhas.append((provider, modelo, edital_nome, str(e)))
            print(f"       ✗ FALHOU antes do xlsx: {e}")
            continue

        if erro_fatal:
            parciais.append((provider, modelo, edital_nome, arq, erro_fatal))
            print(f"       ⚠ PARCIAL — {erro_fatal}\n         salvo em {arq.name}")
        else:
            sucessos.append((provider, modelo, edital_nome, arq))
            print(f"       ✓ ok — salvo em {arq.name}")

print("\n" + "=" * 60)
print("=== SUMÁRIO ===")
print("=" * 60)
print(f"\n✓ Sucessos: {len(sucessos)}")
for p, m, e, a in sucessos:
    print(f"    {p}/{m} × {e}  →  {a.name}")
print(f"\n⚠ Parciais (estouro de orçamento): {len(parciais)}")
for p, m, e, a, err in parciais:
    print(f"    {p}/{m} × {e}  →  {a.name}\n        {err}")
print(f"\n✗ Falhas: {len(falhas)}")
for p, m, e, err in falhas:
    print(f"    {p}/{m} × {e}\n        {err}")


[run]  anthropic/claude-opus-4-7 × cvm (id=2)
      [ 1/50]   ok   8.03s  calls=2  in=13090 (cache   49%)  out= 338  acum=$0.0739
      [ 2/50]   ok    8.6s  calls=2  in=12607 (cache   51%)  out= 373  acum=$0.1463
      [ 3/50]   ok   7.47s  calls=2  in=13317 (cache   48%)  out= 291  acum=$0.2201
      [ 4/50]   ok  12.29s  calls=2  in=12364 (cache   52%)  out= 561  acum=$0.2960
      [ 5/50]   ok   9.63s  calls=2  in=12289 (cache   52%)  out= 418  acum=$0.3679
      [ 6/50]   ok  16.62s  calls=2  in=12346 (cache   52%)  out= 793  acum=$0.4494
      [ 7/50]   ok  11.74s  calls=2  in=12696 (cache   51%)  out= 575  acum=$0.5273
      [ 8/50]   ok  11.17s  calls=2  in=12617 (cache   51%)  out= 504  acum=$0.6030
      [ 9/50]   ok  12.38s  calls=2  in=12292 (cache   52%)  out= 570  acum=$0.6787
      [10/50]   ok   9.22s  calls=2  in=12484 (cache   52%)  out= 405  acum=$0.7512
      [11/50]   ok  10.14s  calls=2  in=13211 (cache   49%)  out= 480  acum=$0.8293
      [12/50]   ok  12.99s  c

## Sumário consolidado

In [9]:
import re

padrao_nome = re.compile(
    r"^(?P<edital>[^_]+)__(?P<provider>[^_]+)__(?P<modelo>.+?)__\d{8}_\d{6}(?:__PARCIAL)?\.xlsx$"
)

linhas = []
for arq in sorted(RESULTADOS_DIR.glob("*.xlsx")):
    m = padrao_nome.match(arq.name)
    if not m:
        continue
    df = pd.read_excel(arq)
    if "cache_read_tokens" not in df.columns:
        df["cache_read_tokens"] = 0
    if "custo_real_usd" not in df.columns:
        df["custo_real_usd"] = df["custo_usd"]

    linhas.append({
        "edital":           m["edital"],
        "provider":         m["provider"],
        "modelo":           m["modelo"],
        "n_perguntas":      len(df),
        "n_erros":          int(df["erro"].notna().sum()),
        "input_tokens":     int(df["input_tokens"].sum()),
        "output_tokens":    int(df["output_tokens"].sum()),
        "cache_read":       int(df["cache_read_tokens"].sum()),
        "cache_pct":        float(df["cache_read_tokens"].sum() / max(df["input_tokens"].sum(), 1)),
        "custo_estim_usd":  round(float(df["custo_usd"].sum()), 4),
        "custo_real_usd":   round(float(df["custo_real_usd"].sum()), 4),
        "latencia_med_s":   round(float(df["latencia_s"].mean()), 2),
        "latencia_p95_s":   round(float(df["latencia_s"].quantile(0.95)), 2),
        "arquivo":          arq.name,
    })

df_sumario = pd.DataFrame(linhas).sort_values(["edital", "custo_real_usd"]).reset_index(drop=True)
df_sumario["cache_pct"] = df_sumario["cache_pct"].map("{:.1%}".format)
df_sumario

,edital,provider,modelo,n_perguntas,n_erros,input_tokens,output_tokens,cache_read,cache_pct,custo_estim_usd,custo_real_usd,latencia_med_s,latencia_p95_s,arquivo
0,bndes,deepseek,deepseek-v4-flash,50,0,651794,21562,1223680,187.7%,0.0973,0.0405,8.89,17.08,bndes__deepseek__deepseek-v4-flash__20260429_2...
1,bndes,openai,gpt-4o-mini,50,0,423484,9378,60416,14.3%,0.0691,0.0646,5.45,10.38,bndes__openai__gpt-4o-mini__20260429_231806.xlsx
2,bndes,openai,gpt-5.4-mini,50,0,332401,8532,84864,25.5%,0.1002,0.0896,2.94,4.73,bndes__openai__gpt-5.4-mini__20260429_235130.xlsx
3,bndes,deepseek,deepseek-v4-pro,50,0,681518,23169,772096,113.3%,1.2665,0.2201,17.69,33.55,bndes__deepseek__deepseek-v4-pro__20260430_001...
4,bndes,openai,gpt-5.4,50,0,416440,8978,169472,40.7%,0.6103,0.5044,5.34,9.58,bndes__openai__gpt-5.4__20260430_143844.xlsx
5,bndes,anthropic,claude-haiku-4-5,50,0,700866,23247,0,0.0%,0.8171,0.8171,15.57,35.46,bndes__anthropic__claude-haiku-4-5__20260430_0...
6,bndes,openai,gpt-5.5,50,0,457896,13135,12160,2.7%,2.6835,2.6531,21.64,37.66,bndes__openai__gpt-5.5__20260430_154956.xlsx
7,bndes,anthropic,claude-sonnet-4-6,50,0,752603,28433,315931,42.0%,2.6843,2.6843,12.58,20.95,bndes__anthropic__claude-sonnet-4-6__20260430_...
8,bndes,anthropic,claude-opus-4-7,50,0,905074,29476,355910,39.3%,5.2623,5.2623,12.08,23.33,bndes__anthropic__claude-opus-4-7__20260430_15...
9,cvm,deepseek,deepseek-v4-flash,50,0,558979,19510,715264,128.0%,0.0837,0.0262,7.82,13.25,cvm__deepseek__deepseek-v4-flash__20260429_234...


## Tabela cruzada — custo real por modelo × edital

In [10]:
df_pivot = (
    df_sumario.assign(modelo_full=lambda d: d["provider"] + "/" + d["modelo"])
              .pivot_table(index="modelo_full", columns="edital",
                          values="custo_real_usd", aggfunc="last")
)
df_pivot["total"] = df_pivot.sum(axis=1)
df_pivot.sort_values("total")

edital,bndes,cvm,petrobras,total
modelo_full,,,,
deepseek/deepseek-v4-flash,0.0405,0.0262,0.0359,0.1026
openai/gpt-4o-mini,0.0646,0.0594,0.0660,0.1900
openai/gpt-5.4-mini,0.0896,0.0872,0.1028,0.2796
deepseek/deepseek-v4-pro,0.2201,0.1899,0.2362,0.6462
openai/gpt-5.4,0.5044,0.4921,0.5265,1.5230
anthropic/claude-haiku-4-5,0.8171,0.7641,1.0032,2.5844
openai/gpt-5.5,2.6531,2.4716,2.9139,8.0386
anthropic/claude-sonnet-4-6,2.6843,2.4376,2.9466,8.0685
anthropic/claude-opus-4-7,5.2623,4.8211,5.9609,16.0443


In [11]:
df_pivot['total'].sum()

np.float64(37.477199999999996)